In [ ]:
import pandas as pd                                  # 판다스 불러오기
import seaborn as sns                                # seaborn 불러오기 (titanic 데이터 다운로드용)

In [ ]:
# 실행 전 준비: VS Code 탐색기에서 마우스로 pandas_prj 폴더 안에 data → raw 폴더를 만들어 두세요.
# seaborn이 없다는 에러가 나면 터미널(.venv 활성화 상태)에서 pip install seaborn 실행

titanic_raw = sns.load_dataset("titanic")            # seaborn 예제 데이터 titanic 다운로드
titanic_raw.to_csv("data/raw/titanic.csv", index=False)  # data/raw 폴더에 csv로 저장
print("저장 완료")

In [ ]:
titanic = pd.read_csv("data/raw/titanic.csv")      # 저장한 파일 불러오기
print(titanic.shape)                                 # (891, 15)가 나오면 성공
print(titanic.head(3))

## 1. 열 선택

In [ ]:
# 1) fare 열 하나 선택 -> 타입 출력
fare = titanic["fare"]
print(type(fare))

# 2) 4개 열 선택 -> 타입과 shape 출력
four = titanic[["survived", "pclass", "sex", "age"]]
print(type(four))
print(four.shape)

# 3) 열 하나를 대괄호 한 겹으로 선택하면 Series, 리스트로 여러 열을 선택하면 DataFrame이 되기 때문이다.

## 2. loc vs iloc

In [ ]:
# 1) loc: 인덱스 라벨 10~15 (끝 포함)
r_loc = titanic.loc[10:15, ["sex", "age"]]
print(r_loc)

# 2) iloc: 행 위치 10:15 (끝 미포함), 열 위치는 sex=2, age=3 (0부터 시작)
print(titanic.columns.tolist())
sex_pos = titanic.columns.get_loc("sex")
age_pos = titanic.columns.get_loc("age")
r_iloc = titanic.iloc[10:15, [sex_pos, age_pos]]
print(r_iloc)

# 3) 행 수 출력
print(len(r_loc), len(r_iloc))
# loc은 라벨 기준이라 끝(15)을 포함해 6행, iloc은 위치 기준이라 끝(15)을 포함하지 않아 5행이다.

## 3. iloc 응용

In [ ]:
print(titanic.iloc[-3:, :4])

## 4. 조건 마스크

In [ ]:
# 1) 나이 10살 미만 마스크
mask = titanic["age"] < 10
print(mask.head(5))

# 2) 10살 미만 승객 수
print(len(titanic[mask]))

## 5. 복합 조건

In [ ]:
# 1) 3등석 & 남성 & 생존
c1 = titanic[(titanic["pclass"] == 3) & (titanic["sex"] == "male") & (titanic["survived"] == 1)]
print(len(c1))

# 2) 1등석 | 요금 100 이상
c2 = titanic[(titanic["pclass"] == 1) | (titanic["fare"] >= 100)]
print(len(c2))

# 3) 혼자 탑승하지 않은 승객
c3 = titanic[~titanic["alone"]]
print(len(c3))

## 6. isin · between

In [ ]:
# 1) Cherbourg 또는 Queenstown 탑승
town = titanic[titanic["embark_town"].isin(["Cherbourg", "Queenstown"])]
print(town.shape)

# 2) 나이 20~30세 (양 끝 포함)
age_both = titanic[titanic["age"].between(20, 30)]
print(age_both.shape)
print(age_both["age"].min(), age_both["age"].max())

# 3) 양 끝 미포함
age_neither = titanic[titanic["age"].between(20, 30, inclusive="neither")]
print(age_neither.shape)

## 7. loc[조건, 열]

In [ ]:
kids = titanic.loc[(titanic["age"] <= 12) & (titanic["pclass"] == 3), ["age", "sex", "survived"]]
print(len(kids))
print(round(kids["survived"].mean(), 2))

## 8. query

In [ ]:
# 1) 변수를 @로 참조
min_fare = 50
q1 = titanic.query("embark_town == 'Cherbourg' and fare >= @min_fare")
print(q1.shape)

# 2) 5-1번 조건을 query로 다시 작성
q2 = titanic.query("pclass == 3 and sex == 'male' and survived == 1")
print(len(q2))
print(len(q2) == len(c1))   # 5-1번 결과와 행 수가 같은지 확인

## 9. 정렬

In [ ]:
# 1) pclass 오름차순, 같은 등급 안에서 fare 내림차순
sorted_df = titanic.sort_values(["pclass", "fare"], ascending=[True, False])
print(sorted_df[["pclass", "fare", "sex", "age"]].head(3))

# 2) 요금 내림차순 정렬 후 iloc[0] vs loc[0]
by_fare = titanic.sort_values("fare", ascending=False)
print(by_fare.iloc[0]["fare"])
print(by_fare.loc[0]["fare"])
# iloc[0]은 정렬 후 맨 앞 행(가장 비싼 요금)이고, loc[0]은 인덱스 라벨이 0인 원래 첫 번째 승객이다.
# 정렬해도 인덱스 라벨은 그대로 유지되기 때문에 두 값이 다르다.

## 10. 종합 — 20~40세 남성, 1·2등석 vs 3등석

In [ ]:
# 1) 남성 & 20~40세 & 1·2등석 -> 4개 열 선택
cond = (
    (titanic["sex"] == "male")
    & titanic["age"].between(20, 40)
    & titanic["pclass"].isin([1, 2])
)
upper = titanic.loc[cond, ["pclass", "age", "fare", "survived"]]

# 2) 행 수, 생존률, 요금 상위 3명
print(len(upper))
print(round(upper["survived"].mean(), 2))
print(upper.sort_values("fare", ascending=False).head(3))

In [ ]:
# 3) 같은 조건을 query로 작성해 행 수 검증
upper_q = titanic.query("sex == 'male' and 20 <= age <= 40 and pclass in [1, 2]")
print(len(upper_q))
print(len(upper_q) == len(upper))

In [ ]:
# 4) 등급만 3등석으로 바꿔 생존률 계산
cond3 = (
    (titanic["sex"] == "male")
    & titanic["age"].between(20, 40)
    & (titanic["pclass"] == 3)
)
third = titanic.loc[cond3, ["pclass", "age", "fare", "survived"]]
print(len(third))
print(round(third["survived"].mean(), 2))

**해석**

20~40세 남성 중 1·2등석 승객의 생존률이 3등석 승객의 생존률보다 뚜렷하게 높게 나타났다. 같은 성별과 연령대인데도 객실 등급에 따라 생존률 차이가 크다는 뜻이다.
다만 이 결과는 등급과 생존의 관계를 보여줄 뿐, 등급이 생존의 원인이라고 단정할 수는 없다.